# Uniform versus Gaussian environmental PV gradient

Compare the original equal-cell ellipse average with the vorticity-shaped spatial weighting, retaining one vector per eddy snapshot.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import esp_pv_tools as ept
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}


In [ ]:
data = ept.load_cache()
paired = ept.paired_to(data)
paired = paired[paired.method.isin(["uniform_0.75","uniform_1"])].copy()
paired["magnitude_ratio"] = paired.PV_grad_mag / paired.reference_PV_grad_mag
paired["direction_change"] = np.abs((paired.PV_grad_theta-paired.reference_PV_grad_theta+180)%360-180)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, method in zip(axes, ["uniform_0.75","uniform_1"]):
    part=paired[paired.method.eq(method)]
    hb=ax.hexbin(part.reference_PV_grad_mag,part.PV_grad_mag,gridsize=45,mincnt=1,bins="log",xscale="log",yscale="log",cmap="viridis")
    limits=np.nanpercentile(np.r_[part.reference_PV_grad_mag,part.PV_grad_mag],[1,99]); ax.plot(limits,limits,"r--")
    ax.set(xlim=limits,ylim=limits,xlabel="ESP-Gaussian frac=2 magnitude",ylabel=f"{method} magnitude",title=method)
fig.suptitle("Matched snapshot comparison"); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
sns.violinplot(data=paired,x="method",y="magnitude_ratio",hue="Cyc",split=True,cut=0,palette=palette,ax=axes[0])
sns.ecdfplot(data=paired,x="direction_change",hue="method",ax=axes[1])
axes[0].axhline(1,color="k",ls="--"); axes[0].set(yscale="log",xlabel="",ylabel="Uniform / Gaussian magnitude")
axes[1].set(xlim=(0,180),xlabel="Direction difference (degrees)"); plt.show()